# Building agents with **v2 Workflows**

A v2 *workflow* is a directed graph — a small state machine — that you author in
YAML and run as a single **interaction**: the caller hands an input to the
**start** node and reads the result from the **end** node.

Between those two boundaries you wire together a handful of node types:

| Node | What it does |
|------|--------------|
| `start` / `end` | the interaction boundary (input in, output out) |
| `llm` | one structured LLM completion |
| `agent` | a multi-step, tool-using agent loop |
| `function` | a single tool call (`python://`, `rest://`, `mcp://`) |
| `if` / `switch` | branch on a simple string expression |

The engine walks the graph, **checkpoints a JSON-serialisable state** after every
node, and records per-node debug data and model statistics through pluggable
**storage** and **task-logger** backends. This walk-through builds those ideas up
one node at a time.

## Setup

We use the SQLite (in-memory) backends so everything here runs locally, and
`openai/gpt-4o-mini` for the LLM nodes. Setting `KAVALAI_DEFAULT_LLM_MODEL` lets
us omit `llm_model` from every graph below.

In [1]:
import os
import dotenv

dotenv.load_dotenv("../.env")

os.environ["KAVALAI_DEFAULT_LLM_MODEL"] = "openai/gpt-4o-mini"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY to run the LLM nodes."

from kavalai.agents.v2.workflow import (
    WorkflowEngine,
    SqliteDataStorage,
    SqliteTaskLogger,
)

/home/timo/projects/kaval.ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. The smallest workflow: `start → llm → end`

A graph is `name` + `data_types` + `nodes`. Two conventions make the boundary
ergonomic: the input is the data type named **`input`**, and the value returned
to the caller is the **`output`** variable named on the `end` node.

`data_types` are JSON-schema fragments compiled to Pydantic models, so every
node's input and output is validated. Inside a prompt you can interpolate the
context with `{{ context.<path> }}`.

In [2]:
greeter_yaml = """
name: Greeter
description: A one-node agent that greets the user by name.
data_types:
  input:
    type: object
    properties:
      user_message: {type: string}
  output:
    type: object
    properties:
      agent_response: {type: string}
nodes:
  - {name: start, type: start, next: reply}
  - name: reply
    type: llm
    prompt: |
      You are a warm, concise greeter.
      Reply to the user in one friendly sentence: {{ context.input.user_message }}
    inputs:
      input: {type: context, value: input}
    output: output
    next: end
  - {name: end, type: end, output: output}
"""

engine = WorkflowEngine.from_yaml(
    greeter_yaml,
    storage=SqliteDataStorage(),
    task_logger=SqliteTaskLogger(),
)
state = await engine.run({"user_message": "Hi, I'm Timo!"})

print(state.output_data["agent_response"])
print("path:", " → ".join(state.trace))

2026-06-17 13:30:33.239 | INFO     | kavalai.agents.v2.workflow.engine:_finish:433 - Workflow 'Greeter' completed (session=1c5f2d04-d4f5-4c22-a048-199c3f067001)


Hi Timo! It's great to meet you!
path: start → reply → end


## 2. The run state is serialisable and persisted

`engine.run(...)` returns a `WorkflowState`: the status, the ordered `trace` of
visited nodes, the full context `data`, and the final `output_data`. It round-trips
through JSON and is **checkpointed to storage after every node**, so a run can be
reloaded and inspected later. The interaction is also recorded as a chat session.

In [3]:
# The entire state serialises to JSON (and back via WorkflowState.from_json).
print(state.to_json()[:240], "…\n")

# It was checkpointed to storage after each node — reload it by run id.
reloaded = await engine.storage.load_state(state.run_id)
print("reloaded status:", reloaded.status, "| nodes:", reloaded.trace, "\n")

# The same backend recorded the conversation turns.
for msg in await engine.storage.get_chat_history(state.session_id):
    print(f"{msg.role:>9}: {msg.content}")

{"workflow_name":"Greeter","status":"completed","current_node":"end","trace":["start","reply","end"],"data":{"input":{"user_message":"Hi, I'm Timo!"},"output":{"agent_response":"Hi Timo! It's great to meet you!"}},"input_data":{"user_messag …

reloaded status: completed | nodes: ['start', 'reply', 'end'] 

     user: Hi, I'm Timo!
assistant: Hi Timo! It's great to meet you!


## 3. Function nodes — calling tools

A `function` node performs exactly one tool call through the `FunctionKernel`,
addressed by URI (`python://`, `rest://`, `mcp://`). Register a Python tool on
the engine's kernel and reference it from the node. The structured return value
is stored in the node's `output` variable and stays in the context for later
nodes to use.

In [4]:
from pydantic import BaseModel
from kavalai.functionkernel import pythontool


class Weather(BaseModel):
    summary: str
    celsius: int


@pythontool
def get_weather(city: str) -> Weather:
    # A real tool would call a weather API; we fake it for the demo.
    return Weather(summary=f"clear skies over {city}", celsius=24)


weather_yaml = """
name: Weather forecaster
description: Looks up weather with a function node, then phrases it with an LLM.
data_types:
  input:
    type: object
    properties:
      user_message: {type: string}
      city: {type: string}
  weather:
    type: object
    properties:
      summary: {type: string}
      celsius: {type: integer}
  output:
    type: object
    properties:
      agent_response: {type: string}
nodes:
  - {name: start, type: start, next: fetch}
  - name: fetch
    type: function
    tool: python://get_weather
    inputs:
      city: {type: context, value: input.city}
    output: weather
    next: phrase
  - name: phrase
    type: llm
    prompt: "Turn the weather reading into a cheerful one-line forecast."
    inputs:
      weather: {type: context, value: weather}
    output: output
    next: end
  - {name: end, type: end, output: output}
"""

engine = WorkflowEngine.from_yaml(weather_yaml, storage=SqliteDataStorage())
engine.kernel.register_python_tool("get_weather", get_weather)

state = await engine.run({"user_message": "weather?", "city": "Tallinn"})
print("tool output kept in context:", state.data["weather"])
print("final:", state.output_data["agent_response"])

2026-06-17 13:30:34.782 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-4o-mini): 104 tokens, 1.33s
2026-06-17 13:30:34.788 | INFO     | kavalai.agents.v2.workflow.engine:_finish:433 - Workflow 'Weather forecaster' completed (session=83db6b7b-82ea-47a2-acde-9a4fb1d5ed5f)


tool output kept in context: {'summary': 'clear skies over Tallinn', 'celsius': 24}
final: Get ready for a beautiful day in Tallinn with clear skies and a lovely 24°C! ☀️


## 4. Branching with `if` and `switch`

Routing nodes evaluate a **simple string expression** against the run context.

- `switch` evaluates `expr`, stringifies it, and matches it against `cases`,
  falling back to `default`.
- `if` branches on a boolean `condition`.

Here an `llm` node classifies the request and a `switch` routes to a specialised
reply. The same graph is run twice and takes a different path each time.

In [5]:
router_yaml = """
name: Support router
description: Classifies a support request and routes to a specialised reply.
data_types:
  input:
    type: object
    properties:
      user_message: {type: string}
  classification:
    type: object
    properties:
      intent: {type: string}
  output:
    type: object
    properties:
      agent_response: {type: string}
nodes:
  - {name: start, type: start, next: classify}
  - name: classify
    type: llm
    prompt: |
      Classify the user's request as exactly one of: refund, technical, other.
      Respond with that single lowercase word in the `intent` field.
    inputs:
      input: {type: context, value: input}
    output: classification
    next: route
  - name: route
    type: switch
    expr: classification.intent
    cases:
      refund: refund_reply
      technical: tech_reply
    default: general_reply
  - name: refund_reply
    type: llm
    prompt: "Empathetically acknowledge the refund request and outline next steps."
    inputs: {input: {type: context, value: input}}
    output: output
    next: end
  - name: tech_reply
    type: llm
    prompt: "Offer one concrete first troubleshooting step for the technical issue."
    inputs: {input: {type: context, value: input}}
    output: output
    next: end
  - name: general_reply
    type: llm
    prompt: "Answer the user's question helpfully and briefly."
    inputs: {input: {type: context, value: input}}
    output: output
    next: end
  - {name: end, type: end, output: output}
"""

for message in ["I want my money back!", "The app crashes on launch"]:
    engine = WorkflowEngine.from_yaml(router_yaml)
    state = await engine.run({"user_message": message})
    print(repr(message))
    print("  intent:", state.data["classification"]["intent"])
    print("  path:  ", " → ".join(state.trace))
    print("  reply: ", state.output_data["agent_response"][:70], "…\n")

2026-06-17 13:30:36.194 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-4o-mini): 89 tokens, 1.27s
2026-06-17 13:30:37.727 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-4o-mini): 123 tokens, 1.46s
2026-06-17 13:30:37.729 | INFO     | kavalai.agents.v2.workflow.engine:_finish:433 - Workflow 'Support router' completed (session=None)


'I want my money back!'
  intent: refund
  path:   start → classify → route → refund_reply → end
  reply:  I completely understand your frustration and I’m here to help you with …



2026-06-17 13:30:39.077 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-4o-mini): 88 tokens, 1.29s
2026-06-17 13:30:40.422 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-4o-mini): 102 tokens, 1.27s
2026-06-17 13:30:40.424 | INFO     | kavalai.agents.v2.workflow.engine:_finish:433 - Workflow 'Support router' completed (session=None)


'The app crashes on launch'
  intent: technical
  path:   start → classify → route → tech_reply → end
  reply:  Check for any available updates for the app and install them, as the c …



### The expression language

`if`/`switch` expressions are evaluated **safely** (an AST whitelist, never
`eval`): comparisons, `and`/`or`/`not`, `in`, arithmetic, and dotted/indexed
access into the context. Unknown names resolve to `None` so guard checks degrade
gracefully. You can call the evaluator directly:

In [6]:
from kavalai.agents.v2.workflow import evaluate_expression, evaluate_bool

ctx = {"state": {"retries": 3, "status": "ok"}, "items": [{"score": 0.9}]}
print(evaluate_expression("state.retries >= 3 and state.status == 'ok'", ctx))
print(evaluate_expression("items[0].score > 0.8", ctx))
print(evaluate_bool("state.status in ['ok', 'done']", ctx))
print(evaluate_expression("missing.value", ctx))  # unknown path -> None

True
True
True
None


## 5. Agent nodes — multi-step tool use

An `agent` node runs the v2 `Agent` loop: it may call tools repeatedly, feeding
results back into its reasoning, before emitting its structured output. Give it a
kernel of tools and a `max_steps` budget.

In [7]:
import datetime


@pythontool
def current_time() -> str:
    return datetime.datetime.now(datetime.timezone.utc).strftime("%A %H:%M UTC")


agent_yaml = """
name: Time-aware assistant
description: An agent that checks the time before greeting the user.
data_types:
  input:
    type: object
    properties:
      user_message: {type: string}
  output:
    type: object
    properties:
      agent_response: {type: string}
nodes:
  - {name: start, type: start, next: assistant}
  - name: assistant
    type: agent
    prompt: "Greet the user appropriately for the current time of day. Use a tool to check the time."
    inputs: {input: {type: context, value: input}}
    output: output
    max_steps: 4
    next: end
  - {name: end, type: end, output: output}
"""

engine = WorkflowEngine.from_yaml(agent_yaml)
engine.kernel.register_python_tool("current_time", current_time)
state = await engine.run({"user_message": "Say hello!"})
print(state.output_data["agent_response"])

2026-06-17 13:30:40.670 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 0/4
2026-06-17 13:30:42.191 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-4o-mini): 769 tokens, 1.52s
2026-06-17 13:30:42.193 | INFO     | kavalai.agents.v2.agent:_call_tool:241 - Calling tool python://current_time with {}
2026-06-17 13:30:42.197 | INFO     | kavalai.agents.v2.agent:prompt:145 - Agent step 1/4
2026-06-17 13:30:43.689 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-4o-mini): 812 tokens, 1.49s
2026-06-17 13:30:43.692 | INFO     | kavalai.agents.v2.workflow.engine:_finish:433 - Workflow 'Time-aware assistant' completed (session=None)


Good morning! How can I assist you today?


## 6. Observability — the task logger

A `TaskLogger` captures per-node debug data and every model call. Logging is
fire-and-forget, so call `flush()` to await the writes. The SQLite logger uses
the same `tasks` / `model_call_stats` shape as the production Postgres backend.

In [8]:
storage = SqliteDataStorage()
tasklog = SqliteTaskLogger()
engine = WorkflowEngine.from_yaml(router_yaml, storage=storage, task_logger=tasklog)
state = await engine.run({"user_message": "How do I reset my password?"})
await tasklog.flush()  # await the fire-and-forget writes

conn = await tasklog._connect()
print("Nodes executed:")
async with conn.execute(
    "SELECT name, node_type, duration_seconds FROM tasks ORDER BY rowid"
) as cur:
    for name, ntype, dur in await cur.fetchall():
        print(f"  {name:<14} {ntype:<8} {dur:.2f}s")

async with conn.execute("SELECT model, total_tokens FROM model_call_stats") as cur:
    rows = await cur.fetchall()
print(f"\nModel calls: {len(rows)} · tokens: {sum(r[1] or 0 for r in rows)}")

2026-06-17 13:30:47.329 | INFO     | kavalai.agents.v2.workflow.engine:_finish:433 - Workflow 'Support router' completed (session=407c66d6-4367-40f0-8527-23ac52fe3525)


Nodes executed:
  classify       llm      2.31s
  tech_reply     llm      1.12s

Model calls: 2 · tokens: 210


## 7. Backends & deterministic testing

Persistence is pluggable behind two interfaces: **`DataStorage`** (agents /
sessions / runs / chat) and **`TaskLogger`** (node logs / model stats). This
notebook used the SQLite in-memory backends; the Postgres backends map onto the
existing `agents`/`sessions`/`runs`/`chat_messages` tables — swap them in without
touching a graph.

To test a workflow **without a live model**, inject a `client_factory`. The
engine calls it as `factory(model, parameters, stats_receiver)`, so you can hand
back a canned client and get fully deterministic, offline runs.

In [9]:
from kavalai.llm_clients.base_client import BaseLlmClient


class StubClient(BaseLlmClient):
    """Returns canned structured output — no network, fully deterministic."""

    def __init__(self, *args, **kwargs):
        super().__init__()

    async def chat_completions(self, *, chat_history, response_model=None):
        values = {
            name: ("refund" if name == "intent" else "Stubbed reply.")
            for name in response_model.model_fields
        }
        return response_model(**values)


engine = WorkflowEngine.from_yaml(
    router_yaml, client_factory=lambda *a, **k: StubClient()
)
state = await engine.run({"user_message": "anything at all"})
print("deterministic path:", " → ".join(state.trace))  # always routes via 'refund'

2026-06-17 13:30:47.460 | INFO     | kavalai.agents.v2.workflow.engine:_finish:433 - Workflow 'Support router' completed (session=None)


deterministic path: start → classify → route → refund_reply → end


## Recap

- A workflow is a **DAG of typed nodes** authored in YAML; the caller talks to
  the `start`/`end` boundary.
- `llm`, `agent`, and `function` nodes do the work; `if`/`switch` route on safe
  string expressions.
- Every run produces a **JSON-serialisable `WorkflowState`**, checkpointed and
  logged through pluggable backends.
- Inject a `client_factory` for fast, deterministic tests.

A complete, runnable graph lives at
[`examples/v2_workflow_support_agent.yaml`](../examples/v2_workflow_support_agent.yaml).